# 01 — Quickstart

Goal: in ~5 minutes, import `rldx`, load a pre-trained checkpoint,
feed a dummy observation through the policy, and read out an action.

This notebook requires a CUDA-capable GPU. The first run downloads
the checkpoint (~15 GB) into your HuggingFace cache.


## 1. Environment check


In [ ]:
import rldx
import torch

print('rldx    :', rldx.__version__)
print('torch   :', torch.__version__)
print('cuda?   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu    :', torch.cuda.get_device_name(0))


## 2. Load a pre-trained checkpoint

`RLDXPolicy` is the main inference entry point. Under the hood it:

1. Downloads the checkpoint (if `model_path` is a HuggingFace repo id),
2. Loads the RLDX-1 model via `AutoModel.from_pretrained(..., torch_dtype=torch.bfloat16)`,
3. Loads the `RLDXProcessor` from the `processor/` subdirectory of the checkpoint,
4. Attaches the embodiment-specific modality config.


In [ ]:
from rldx.data.embodiment_tags import EmbodimentTag
from rldx.policy.rldx_policy import RLDXPolicy, RLDXSimPolicyWrapper

# Default checkpoint for the quick-start inference call.
MODEL = 'RLWRLD/RLDX-1-FT-ROBOCASA'

policy = RLDXPolicy(
    embodiment_tag=EmbodimentTag.GENERAL_EMBODIMENT,
    model_path=MODEL,
    device='cuda:0',
    strict=True,
)
print('loaded:', type(policy.model).__name__)
print('processor:', type(policy.processor).__name__)
print('modality keys:', list(policy.modality_configs.keys()))


## 3. Build a dummy observation

For this quickstart we target the `robocasa_panda_omron` embodiment:
three 256x256 RGB views (`left_view`, `right_view`, `wrist_view`),
a small state vector, and a natural-language instruction.

`RLDXSimPolicyWrapper` accepts the flat observation format used by
the simulation environments: keys are `video.<name>`, `state.<name>`,
and the language field is `annotation.human.action_tasks` (or `task`).
Video arrays are `uint8` shaped `(B, T, H, W, 3)`; state arrays are
`float32` shaped `(B, T, D)`.


In [ ]:
import numpy as np

B, Tv, Ts, H, W = 1, 4, 1, 256, 256
rng = np.random.default_rng(0)

obs = {
    # Video — one frame per view.
    'video.left_view':  rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),
    'video.right_view': rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),
    'video.wrist_view': rng.integers(0, 255, (B, Tv, H, W, 3), dtype=np.uint8),

    # State — one timestep, matching the robocasa state schema.
    'state.end_effector_position_relative': np.zeros((B, Ts, 3), dtype=np.float32),
    'state.end_effector_rotation_relative': np.zeros((B, Ts, 4), dtype=np.float32),
    'state.gripper_qpos':                   np.zeros((B, Ts, 2), dtype=np.float32),
    'state.base_position':                  np.zeros((B, Ts, 3), dtype=np.float32),
    'state.base_rotation':                  np.zeros((B, Ts, 4), dtype=np.float32),

    # Language — one string per batch item.
    'annotation.human.action.task_description': ('pick up the mug',),
}
print({k: (v.shape, v.dtype) if hasattr(v, 'shape') else v for k, v in obs.items()})


## 4. Run inference

Wrap the policy in `RLDXSimPolicyWrapper` so it accepts the flat
observation format we just built. Then call `get_action`. The
returned `actions` is a dict keyed by the action modalities defined
in the checkpoint's modality config; each value has the model's
action horizon as its first dimension.


In [ ]:
sim_policy = RLDXSimPolicyWrapper(policy, strict=True)

actions, info = sim_policy.get_action(obs)

print('action keys:', list(actions.keys()))
for k, v in actions.items():
    print(f'  {k:32s} shape={tuple(v.shape)} dtype={v.dtype}')


## 5. Next steps

- Understand the dataset layout this model expects → [`02_dataset_preparation.ipynb`](02_dataset_preparation.ipynb)
- Finetune on your own data → [`03_finetuning.ipynb`](03_finetuning.ipynb)
- Run inference as a network service → [`04_inference_server.ipynb`](04_inference_server.ipynb)
